# 训练营验收体系：技能树与作品集

> 本 Notebook 对应文章：`011_S0_训练营验收体系：技能树与作品集[通用][入门].md`
>
> 目标：逐步执行文章中的所有代码，验证可行性

## 环境准备

首次运行时，请先安装依赖包

In [ ]:
# 安装依赖（首次运行时取消注释）
# !pip install scanpy squidpy matplotlib

In [ ]:
import scanpy as sc
import squidpy as sq

# Load data
adata = sq.datasets.visium_hne_adata()

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Visualize QC metrics
sc.pl.violin(
    adata, 
    ['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    jitter=0.4, 
    multi_panel=True
)

# Spatial distribution of QC metrics
sq.pl.spatial_scatter(
    adata,
    color=['total_counts', 'n_genes_by_counts'],
    size=1.5
)

# Filter based on QC thresholds
print(f"Before filtering: {adata.n_obs} spots")

sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_cells(adata, max_counts=10000)
adata = adata[adata.obs['pct_counts_mt'] < 20].copy()

print(f"After filtering: {adata.n_obs} spots")
print(f"Removed: {adata.n_obs} spots")

**常见错误**：
- ❌ 使用固定阈值（如 min_counts=1000）而不检查数据分布
- ❌ 过滤后不检查空间分布是否出现空洞
- ❌ 没有记录过滤参数和理由

**进阶挑战**：
- 对比不同过滤阈值对下游分析的影响
- 识别组织边缘的低质量 spot
- 处理批次间 QC 指标差异

## 里程碑：学习路径与阶段目标

### 里程碑设计原则

里程碑（Milestone）是技能树的**检查点**，每个里程碑对应一个可交付的成果。

**设计原则**：
1. **可验证**：有明确的输出物（代码、图表、报告）
2. **递进式**：后一个里程碑依赖前一个的能力
3. **实战化**：每个里程碑解决一个真实问题

### 训练营里程碑地图

### 里程碑验收清单

以 **M1: 单样本分析** 为例：

**输入**：
- Visium 数据（表达矩阵 + 空间坐标 + 组织图像）

**输出**：
- [ ] QC 报告（小提琴图 + 空间分布图）
- [ ] 过滤后的数据对象（.h5ad 文件）
- [ ] 聚类结果（UMAP + 空间分布）
- [ ] 标记基因列表（每个 cluster 的 top 10 基因）

**验收标准**：
1. ✅ QC 指标分布合理（无异常峰）
2. ✅ 过滤阈值有明确依据
3. ✅ 聚类结果在空间上连续（非随机分布）
4. ✅ 标记基因与组织结构匹配（如上皮、基质、免疫）

**时间预估**：
- 新手：8-10 小时
- 有 scRNA-seq 经验：4-6 小时

**常见卡点**：
- 不知道如何设置过滤阈值 → 参考 010 号文章的参数选择方法
- 聚类结果空间上很碎 → 调整分辨率参数或使用空间约束聚类
- 标记基因不明显 → 检查归一化方法和差异分析参数

## 作品集：能力的可视化证明

### 作品集结构

作品集（Portfolio）是你能力的**直接证明**，比简历上的"熟悉空间转录组分析"更有说服力。

**作品集包含三部分**：

### 作品集模板

#### 模板 1：单样本空间域分析

**项目名称**：Human Brain Spatial Domain Identification

**数据来源**：10x Genomics Visium - Human Brain (FFPE)

**分析目标**：识别大脑皮层的空间域并解析其基因表达特征

**关键结果**：
1. 识别出 7 个空间域（对应皮层分层结构）
2. 发现 120 个空间可变基因
3. 验证了已知的层特异性标记基因（如 Layer 5 的 PCP4）

**技术亮点**：
- 使用 Leiden 聚类 + 空间约束优化域边界
- 对比了 3 种空间可变基因检测方法（Moran's I, SpatialDE, SPARK）
- 构建了完整的证据链（诊断图 + 负对照 + 敏感性分析）

**代码示例**：

In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Load data
adata = sq.datasets.visium_hne_adata()

# Standard preprocessing
sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Spatial domain identification
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=6)
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.leiden(adata, resolution=0.5, key_added='spatial_domain')

# Visualize spatial domains
sq.pl.spatial_scatter(
    adata,
    color='spatial_domain',
    size=1.5,
    title='Spatial Domains'
)

# Identify spatially variable genes
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    n_perms=100,
    n_jobs=1
)

# Get top SVGs
svg_results = adata.uns['moranI']
top_svgs = svg_results.sort_values('I', ascending=False).head(20)
print("Top 20 Spatially Variable Genes:")
print(top_svgs[['I', 'pval_norm']])

# Visualize top SVGs
sq.pl.spatial_scatter(
    adata,
    color=top_svgs.index[:4].tolist(),
    size=1.5,
    ncols=2
)

**证据链**：
1. **诊断图**：空间域在 UMAP 和空间上都连续
2. **对照**：随机打乱空间坐标后，Moran's I 显著下降
3. **敏感性**：分辨率从 0.3 到 0.8，核心域保持稳定
4. **边界声明**：该方法适用于组织结构清晰的样本，对高度异质性肿瘤可能需要更高分辨率数据

**GitHub 链接**：`github.com/your-username/spatial-domain-analysis`

#### 模板 2：多样本细胞通讯分析

**项目名称**：Tumor-Immune Cell Communication in Spatial Context

**数据来源**：Visium - Human Breast Cancer (3 samples)

**分析目标**：比较肿瘤核心与边缘的细胞通讯模式差异

**关键结果**：
1. 肿瘤边缘的免疫-肿瘤通讯显著高于核心区域
2. 识别出 15 对显著的配体-受体互作
3. 发现 PD-L1/PD-1 轴在肿瘤边缘富集

**技术亮点**：
- 使用 Cell2location 进行细胞类型反卷积
- 结合空间距离计算细胞通讯概率
- 多样本批次效应校正（Harmony）

**代码示例**：

In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd

# Load deconvolution results (假设已完成 Cell2location)
adata = sc.read_h5ad('adata_with_celltype_proportions.h5ad')

# Define tumor core vs edge based on spatial location
# (简化示例，实际需要更复杂的区域定义)
adata.obs['region'] = 'core'
# Mark edge spots (示例逻辑)
edge_spots = adata.obs['tumor_proportion'] > 0.3
adata.obs.loc[edge_spots, 'region'] = 'edge'

# Cell-cell communication analysis
sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key='dominant_celltype',
    copy=False
)

# Extract significant interactions
ligrec_results = adata.uns['ligrec']
sig_interactions = ligrec_results[ligrec_results['pvalue'] < 0.05]

# Compare core vs edge
core_data = adata[adata.obs['region'] == 'core']
edge_data = adata[adata.obs['region'] == 'edge']

print(f"Significant interactions in core: {len(sig_interactions)}")
print(f"Significant interactions in edge: {len(sig_interactions)}")

# Visualize key ligand-receptor pairs
sq.pl.ligrec(
    adata,
    cluster_key='dominant_celltype',
    source_groups=['Tumor'],
    target_groups=['T_cell', 'Macrophage']
)

**证据链**：
1. **诊断图**：细胞类型反卷积的 QC 指标（ELBO 收敛曲线）
2. **对照**：随机打乱细胞类型标签后，通讯信号消失
3. **敏感性**：使用不同的距离阈值（50 µm, 100 µm, 150 µm），核心互作保持显著
4. **边界声明**：该分析基于 spot 级别的反卷积，无法解析单细胞级别的通讯

**GitHub 链接**：`github.com/your-username/tumor-immune-communication`

### 作品集验收清单

在提交作品集前，确保满足以下标准：

**代码质量**：
- [ ] 代码可以直接运行（提供测试数据或下载链接）
- [ ] 设置了随机种子（保证可复现）
- [ ] 有清晰的注释说明每个步骤的目的
- [ ] 参数选择有明确依据

**分析质量**：
- [ ] 包含完整的证据链（诊断图 + 对照 + 敏感性 + 边界声明）
- [ ] QC 指标在合理范围内
- [ ] 结果有生物学解释
- [ ] 说明了方法的适用范围和局限性

**可视化质量**：
- [ ] 图表标题、轴标签、图例使用英文
- [ ] 配色方案清晰（色盲友好）
- [ ] 分辨率足够（至少 300 DPI）
- [ ] 有图例说明（Figure Legend）

**文档质量**：
- [ ] README 包含数据来源、环境配置、运行方法
- [ ] 分析报告有清晰的章节结构
- [ ] 关键结果有截图或输出示例
- [ ] 提供了环境配置文件（requirements.txt 或 environment.yml）

## 如何使用验收体系

### 新手路径（0-3 个月）

**目标**：完成 M0-M2，建立基础能力

**学习计划**：
1. Week 1-2：环境搭建 + 标准流程
   - 阅读：002（完整流程）、003（工具上手）、010（复现标准）
   - 实战：复现 Scanpy tutorial
   - 验收：提交 M0 作品

2. Week 3-5：单样本分析
   - 阅读：质量控制、归一化、聚类相关文章
   - 实战：分析 10x 官方数据集
   - 验收：提交 M1 作品

3. Week 6-8：空间分析
   - 阅读：空间可变基因、空间域识别
   - 实战：发现空间特异性模式
   - 验收：提交 M2 作品

**验收标准**：
- 能独立完成单样本的标准分析流程
- 能解释每个步骤的目的和参数选择
- 能识别常见的数据质量问题

### 进阶路径（3-6 个月）

**目标**：完成 M3-M4，掌握高级方法

**学习计划**：
1. Week 9-11：细胞类型反卷积
   - 阅读：Cell2location、RCTD、Tangram 相关文章
   - 实战：推断细胞类型空间分布
   - 验收：提交 M3 作品

2. Week 12-14：细胞通讯分析
   - 阅读：CellChat、COMMOT、Giotto 相关文章
   - 实战：识别配体-受体互作
   - 验收：提交 M4 作品

**验收标准**：
- 能选择合适的方法解决复杂问题
- 能构建完整的证据链
- 能评估方法的适用性和局限性

### 专家路径（6-12 个月）

**目标**：完成 M5，达到可发表水平

**学习计划**：
1. Week 15-18：多样本整合与批次效应校正
   - 阅读：Harmony、Seurat integration、LIGER 相关文章
   - 实战：整合多个数据集
   - 验收：批次效应得到有效控制

2. Week 19-22：参数优化与敏感性分析
   - 阅读：参数选择、稳健性验证相关文章
   - 实战：系统评估参数对结果的影响
   - 验收：结论在合理参数范围内稳定

3. Week 23-26：可复现报告生成
   - 阅读：010（复现标准）、工程化交付相关文章
   - 实战：整理完整的分析流程
   - 验收：提交 M5 作品

**验收标准**：
- 分析流程完全可复现
- 证据链完整且逻辑严密
- 结果达到可发表水平

## 验收清单：你准备好了吗

### Level 1: 环境搭建
- [ ] 成功安装 Python 3.8+
- [ ] 安装 Scanpy 和 Squidpy
- [ ] 运行官方 tutorial 无报错
- [ ] 能生成基本的可视化图表

### Level 2: 标准流程
- [ ] 能计算并可视化 QC 指标
- [ ] 能根据数据分布设置过滤阈值
- [ ] 能完成归一化和降维
- [ ] 能进行聚类并识别标记基因

### Level 3: 空间分析
- [ ] 能检测空间可变基因
- [ ] 能识别空间域
- [ ] 能进行邻域富集分析
- [ ] 能解释空间模式的生物学意义

### Level 4: 高级方法
- [ ] 能进行细胞类型反卷积
- [ ] 能推断细胞通讯
- [ ] 能整合多个样本
- [ ] 能处理批次效应

### Level 5: 工程化交付
- [ ] 能进行参数优化
- [ ] 能构建完整的证据链
- [ ] 能生成可复现报告
- [ ] 能评估方法的局限性

## 常见问题

**Q1: 我需要按顺序完成所有里程碑吗？**

不一定。如果你已经有 scRNA-seq 分析经验，可以快速通过 M0-M1，直接从 M2 开始。但建议至少完成一次完整的单样本分析，确保理解空间数据的特殊性。

**Q2: 作品集需要使用自己的数据吗？**

不需要。使用公开数据集（如 10x Genomics 官方数据）完全可以。重点是展示你的分析能力和思考过程，而不是数据的新颖性。

**Q3: 如何判断自己的分析质量？**

参考证据链四件套：
1. 诊断图显示数据质量良好
2. 负对照验证方法的特异性
3. 敏感性分析显示结论稳健
4. 边界声明明确方法的适用范围

如果这四点都满足，你的分析质量已经超过大部分发表的文章。

**Q4: 验收体系会更新吗？**

会。随着新方法和新平台的出现，技能树会持续更新。但核心能力（QC、归一化、空间分析、证据链）是不变的。

**Q5: 完成所有里程碑需要多长时间？**

- 新手（无编程经验）：6-12 个月
- 有 Python 经验：3-6 个月
- 有 scRNA-seq 经验：1-3 个月

时间不是重点，重点是每个里程碑都达到验收标准。

## 下一步

现在你已经了解了训练营的验收体系，建议：

1. **评估当前水平**：对照技能树，标记已掌握的技能点
2. **选择起点**：根据评估结果，选择合适的里程碑开始
3. **制定计划**：为每个里程碑设定时间目标
4. **开始实战**：选择一个数据集，开始你的第一个作品

记住：**作品集是能力的最好证明**。与其在简历上写"熟悉空间转录组分析"，不如展示一个完整的分析项目。

下一篇文章（012）将介绍**如何选择合适的公开数据集进行练习**，包括数据质量评估、平台选择、生物学问题匹配等内容。

---

**本文要点**：
- 技能树定义了"会什么"，分为 5 个层级
- 里程碑定义了"到哪了"，每个里程碑有明确的验收标准
- 作品集定义了"能做什么"，是能力的可视化证明
- 验收体系的核心是可验证、可追踪、可展示

**验收清单**：
- [ ] 理解技能树的 5 个层级
- [ ] 知道每个里程碑的验收标准
- [ ] 了解作品集的三个组成部分
- [ ] 能评估自己当前的技能水平
- [ ] 制定了个人学习计划